# Lunden's Analysis of Movie Data

### Main RQ
What is the relationship between IMDB rating and income (domestic vs foreign vs internationl, gross vs %)

### Statistical Requirements
- Descriptive Statistics
- Inferential statistics
- Graphical Analysis
- Comparative Analysis
- Mulivariate Analysis
- Synthesis
- Documentation
- Reporting & Interpretation


In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.simplefilter(action='ignore')
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd

### Descriptive Statistics

In [39]:
movies = pd.read_csv('movies.csv')
print(movies.columns)

Index(['title', 'originalTitle', 'isAdult', 'runtimeMinutes', 'genres',
       'IMDBavgRating', 'numVotes', 'rank', 'worldwideGross', 'domesticGross',
       'domestic%', 'foreignGross', 'foreign%', 'year', 'originalLang',
       'productionCountries', 'main_genre', 'rating_category'],
      dtype='object')


In [40]:
movies.describe()

,isAdult,runtimeMinutes,IMDBavgRating,numVotes,rank,worldwideGross,domesticGross,domestic%,foreignGross,foreign%,year
count,7004.000000,7004.000000,7004.000000,7.004000e+03,7004.000000,7.004000e+03,7.004000e+03,7004.00000,7.004000e+03,7004.000000,7004.000000
mean,0.000714,104.654055,6.254326,8.545767e+04,101.490720,1.137223e+08,4.236133e+07,34.84486,7.136074e+07,65.153998,2012.621074
std,0.026711,28.926183,1.147439,1.812764e+05,56.298787,1.911045e+08,7.372391e+07,30.07744,1.271558e+08,30.076601,6.883668
min,0.000000,0.000000,1.200000,5.000000e+00,1.000000,4.693664e+06,0.000000e+00,0.00000,0.000000e+00,0.000000,2001.000000
25%,0.000000,92.000000,5.600000,4.117500e+02,54.000000,2.562145e+07,1.640742e+05,0.30000,1.480998e+07,42.900000,2007.000000
50%,0.000000,104.000000,6.400000,1.043750e+04,102.000000,4.841293e+07,1.830439e+07,36.60000,3.025607e+07,63.400000,2013.000000
75%,0.000000,119.000000,7.000000,9.189450e+04,150.000000,1.126836e+08,5.110049e+07,57.10000,7.030778e+07,99.600000,2019.000000
max,1.000000,700.000000,10.000000,2.985446e+06,200.000000,2.799439e+09,9.366622e+08,100.00000,1.993811e+09,100.000000,2024.000000


In [41]:
def print_median(df):
    '''Prints the median of columns with numeric data types'''

    for column in df.select_dtypes(include=[np.number]).columns:
        print(f'{column}: {df[column].median()}')

print_median(movies)

isAdult: 0.0
runtimeMinutes: 104.0
IMDBavgRating: 6.4
numVotes: 10437.5
rank: 102.0
worldwideGross: 48412933.0
domesticGross: 18304386.5
domestic%: 36.6
foreignGross: 30256068.0
foreign%: 63.4
year: 2013.0


In [ ]:
def print_grouped_stats(df):
    '''Prints the grouped stats of columns with numeric data types by rating category'''
    for column in df.select_dtypes(include=[np.number]).columns:
        print(f"{column} stats by rating category")
        print(f"{df.groupby('rating_category')[column].describe()}\n")

print_grouped_stats(movies)

isAdult stats by rating category
                  count      mean       std  min  25%  50%  75%  max
rating_category                                                     
High             1728.0  0.000579  0.024056  0.0  0.0  0.0  0.0  1.0
Low               310.0  0.000000  0.000000  0.0  0.0  0.0  0.0  0.0
Medium           4966.0  0.000805  0.028372  0.0  0.0  0.0  0.0  1.0

runtimeMinutes stats by rating category
                  count        mean        std  min   25%    50%    75%    max
rating_category                                                               
High             1728.0  110.563657  33.633029  0.0  96.0  114.0  130.0  280.0
Low               310.0   91.416129  32.843126  0.0  83.0   92.0  105.0  163.0
Medium           4966.0  103.424084  26.362317  0.0  92.0  103.0  116.0  700.0

IMDBavgRating stats by rating category
                  count      mean       std  min  25%  50%  75%   max
rating_category                                                      
High  

### Interpretation of Descriptive Statistics

The first generated dataframe presents general descriptive statistics of the movies data, displaying useful metrics such as mean, standard devation, and the range of values for each numeric feature of the data. I also created print_median() functions to display the median values of each numeric feature. I then grouped the data by rating category (high/medium/low) and compared the descriptive statistics of this dataframe to the original dataframe.

*isAdult*\
This feature is a boolean representation of the adult movie categorization. The mean value is 0.00071, which  means not many movies in this data are adult films. When grouped by rating category, I find that there are adult movies in the high and medium rated categories and not the low rated category.

*runtimeMinutes*\
The mean runtime of movies in this data is 104.7 minutes and the standard deviation is 28.9 minutes. This tells use that there is a large range of runtimes. High rated movies have a higher mean runtime than medium, and medium rated movies higher than low rated movies.

*IMDBavgRating*\
The mean rating 6.3 and the standard deviation is 1.1. From this I can determine that a majority of movies are rated between 5 and 7. The mean average ratings for high rated movies and medium rated movies are close to the full sample mean.

*Income metrics*\
The gross income means for films (worldwide, foreign, and domestic) are in the millions, and the standard deviation are just as large. There is significant variation in these metrics. The median values for each type of gross income are significantly lower than the mean values, suggesting that there is a small number of movies that are making significantly more money than the others. High rated movies have higher mean gross income levels than medium and low rated movies. An interesting finding is that medium and low rated movies have a similar mean foreign gross income level. Each rating category has similar proportions of foreign to domestic income (63:37)

### Inferential Statistics

*Sub RQs*
1. Is there a linear relationship between IMDB rating and gross income? What if I separate by rating category?
2. Do highly rated movies generate more revenue than medium rated movies?
3. Do medium rated movies generate more revenue than low rated movies?

*Testing Methods*
- ANOVA: I will use this to compare gross income statistics between rating categories (high vs medium vs low)
- Regression Analysis: I will use this to determine if there is a significant relationship between rating and income level.

In [ ]:
def find_corr(df, target='IMDBavgRating', columns=None):
    '''Finds the correlation between the target and specified columns''' 
    
    for column in columns:
        print(f"Correlation between {column} and rating")
        print(f"{df[target].corr(df[column])}\n")


find_corr(movies, columns=['domesticGross', 'worldwideGross', 'foreignGross'])

Correlation between domesticGross and rating
0.12208548145052531

Correlation between worldwideGross and rating
0.12235400956470854

Correlation between foreignGross and rating
0.11310184128713203



**Interpretation**\
I found the correlation between IMDB average rating and gross incomes, and I find a weak positive correlation between rating and each type of gross income. I expected there to be a stronger correlation, as I expect highly rated movies to make more money.

In [44]:
low_ratings = movies[movies['rating_category'] == 'Low']
mid_ratings = movies[movies['rating_category'] == 'Medium']
high_ratings = movies[movies['rating_category'] == 'High']

groups = {"low rated": low_ratings, "mid rated": mid_ratings, "high rated": high_ratings}

for k, v in groups.items():
    print(k)
    find_corr(v, columns=['domesticGross', 'worldwideGross', 'foreignGross'])


low rated
Correlation between domesticGross and rating
-0.03639618093610432

Correlation between worldwideGross and rating
-0.06145625544629724

Correlation between foreignGross and rating
-0.0719194647850729

mid rated
Correlation between domesticGross and rating
0.07477188263533831

Correlation between worldwideGross and rating
0.0813949208088475

Correlation between foreignGross and rating
0.07689097613294835

high rated
Correlation between domesticGross and rating
0.0851073097020312

Correlation between worldwideGross and rating
0.10188714778508431

Correlation between foreignGross and rating
0.10662931715860635



**Interpretation**\
After subsetting the data by rating category, I am still not finding a strong correlation between rating and gross income. The correlation between each rating category and type of gross income is weak and positive.

In [ ]:
def generate_ols_model(df, target='IMDBavgRating', columns=None):
    '''Generates an OLS model for the target and specified columns'''
    for column in columns:
        model = smf.ols(f"{column} ~ {target}", data=df).fit()
        print(model.summary())

for k, v in groups.items():
    print(f"{k} OLS model\n")
    generate_ols_model(v, columns=['domesticGross', 'worldwideGross', 'foreignGross'])

low rated OLS model

                            OLS Regression Results                            
Dep. Variable:          domesticGross   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                 -0.002
Method:                 Least Squares   F-statistic:                    0.4085
Date:                Mon, 17 Mar 2025   Prob (F-statistic):              0.523
Time:                        22:15:07   Log-Likelihood:                -5996.2
No. Observations:                 310   AIC:                         1.200e+04
Df Residuals:                     308   BIC:                         1.200e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept      4.782e+07 

**Interpretation**\
In the above cell, I created linear regression models to confirm what I found earlier about the correlations. the R-squared values are very low, meaning the model explains very little of the variance in gross income. Furthermore, the F-statistc for each model is high, meaning the results aren't statistically significant. A linear model doesn't quite explain the relationship between IMDB average rating and gross income.

In [ ]:
def generate_anova(df, columns=None):
    '''Generates an ANOVA table for the specified columns'''
    for column in columns:
        model = smf.ols(f"{column} ~ rating_category", data=df).fit()
        table = sm.stats.anova_lm(model, typ=2)
        print(table)

generate_anova(movies, columns=['domesticGross', 'worldwideGross', 'foreignGross'])


                       sum_sq      df          F        PR(>F)
rating_category  5.525403e+17     2.0  51.563677  5.881829e-23
Residual         3.751027e+19  7001.0        NaN           NaN
                       sum_sq      df          F        PR(>F)
rating_category  3.500224e+18     2.0  48.571836  1.123388e-21
Residual         2.522559e+20  7001.0        NaN           NaN
                       sum_sq      df          F        PR(>F)
rating_category  1.272585e+18     2.0  39.789539  6.562906e-18
Residual         1.119561e+20  7001.0        NaN           NaN


**Interpretation**\
Based on the ANOVA test, I find that there is a significant difference in mean gross income level between different rating categories. I will next use Tukey's honestly significant differences (HSD) to determine which rating categories are different from each other

In [ ]:
def generate_HSD(df, columns=None):
    '''Generates a Tukey HSD test for the specified columns'''
    for column in columns:
        model = pairwise_tukeyhsd(df[column], df['rating_category'])
        print(model.summary())

generate_HSD(movies, columns=['domesticGross', 'worldwideGross', 'foreignGross'])

          Multiple Comparison of Means - Tukey HSD, FWER=0.05           
group1 group2    meandiff    p-adj      lower          upper      reject
------------------------------------------------------------------------
  High    Low -22514668.2327    0.0 -33098407.1154 -11930929.3501   True
  High Medium -20459250.5958    0.0 -25251698.1629 -15666803.0287   True
   Low Medium   2055417.6369 0.8809  -7989767.1824  12100602.4562  False
------------------------------------------------------------------------
          Multiple Comparison of Means - Tukey HSD, FWER=0.05           
group1 group2    meandiff    p-adj      lower          upper      reject
------------------------------------------------------------------------
  High    Low -53627559.8721    0.0 -81073916.8517 -26181202.8925   True
  High Medium  -51736909.295    0.0 -64164958.4174 -39308860.1726   True
   Low Medium   1890650.5771 0.9842 -24159097.2042  27940398.3584  False
---------------------------------------------------

**Interpretation**

*Null Hypothesis*\
There is no difference in worldwide, domestic, or foreign gross income between high, medium, and low rated movies.

*Alternative Hypothesis*\
There is a difference in worldwide, domestic, or foreign gross income between high, medium, and low rated movies.

*p-value*\
0.05

*Results*\
Based on the HSD model, I find that I can reject the null hypothesis that high-rated movies and medium-rated movies have no difference in mean worldwide, domestic, and foreign gross incomes. I can also reject the null hypothesis that high-rated movies and low-rated movies have no difference in average gross worldwide, domestic, and foreign gross income. However, I cannot reject the null hypothesis that medium-rated movies and low-rated movies have statistically different incomes.